# Exercise 2.3 — Meeting-Notes Prompt Helper
### *Where your code and your AI tools shake hands*

This one's a little different — and delightfully sneaky, as promised. Instead of Python doing *all* the work, we're going to use Python to take messy, shorthand meeting notes and turn them into a clean, structured **prompt** that you can hand to an AI assistant to generate a proper summary, action list, or follow-up email.

**Why bother with code here, when the AI does the "smart" part anyway?**

Because "garbage in, garbage out" applies to AI just as much as to any other tool. If you paste raw shorthand notes into a chatbot, you'll get an inconsistent, so-so summary. If you first use a tiny bit of code to reliably pull out attendees, action items, and open questions into a clean structure — *then* hand that structured prompt to the AI — you get a dramatically better, more consistent result, every single time. You're not replacing the AI; you're setting it up to succeed.

## What you'll practise
- Tying together **everything from this course so far**: strings, loops, dictionaries, conditionals
- A first taste of **prompt engineering** — designing input that gets better output

## The scenario
`raw_notes.txt` contains real shorthand meeting notes, the way most people actually type them live in a meeting. Let's parse it, structure it, and build a clean prompt.

## Step 1: Look at the raw notes

In [1]:
with open("raw_notes.txt") as f:
    raw_text = f.read()

print(raw_text)


proj: website redesign
attendees: alex, priya, marcus, farah
discussed - homepage layout still not final, priya to send 2 options by fri
budget - slightly over, need approval from finance for extra 3k for stock photos
timeline - launch pushed from aug 15 to aug 29, dev team needs more QA time
action - marcus to update timeline doc
action - farah to email finance re: budget approval
action - alex to review priya's 2 layout options by mon
open question - do we need a separate mobile design or is responsive enough?
next meeting - next fri 2pm



These notes follow a loose pattern: each line is `label - content` or `label: content`. That's messy, but *consistent enough* for code to work with. This is a common situation in real note-taking - not perfectly structured, but not total chaos either.

## Step 2: Split into lines and separate label from content

In [2]:
lines = raw_text.strip().split("\n")
print(f"Total lines: {len(lines)}")
for line in lines[:5]:
    print(repr(line))


Total lines: 10
'proj: website redesign'
'attendees: alex, priya, marcus, farah'
'discussed - homepage layout still not final, priya to send 2 options by fri'
'budget - slightly over, need approval from finance for extra 3k for stock photos'
'timeline - launch pushed from aug 15 to aug 29, dev team needs more QA time'


In [3]:
def split_label(line):
    # Some lines use ":" and some use "-" as the separator. Handle both.
    if ":" in line:
        label, content = line.split(":", 1)   # split only on the FIRST ":"
    elif "-" in line:
        label, content = line.split("-", 1)
    else:
        label, content = "note", line
    return label.strip(), content.strip()

# test it
for line in lines[:4]:
    print(split_label(line))


('proj', 'website redesign')
('attendees', 'alex, priya, marcus, farah')
('discussed', 'homepage layout still not final, priya to send 2 options by fri')
('budget', 'slightly over, need approval from finance for extra 3k for stock photos')


**Why `split(":", 1)` and not just `split(":")`?** The `1` means "split at most once." Without it, a line like `next meeting - next fri 2pm` with a colon inside the content would get split into three pieces instead of two, and break our unpacking into `label, content`. Setting a limit on `.split()` is a small habit that prevents a surprisingly common bug.

## Step 3: Group lines into categories

In [4]:
structured = {
    "project": [],
    "attendees": [],
    "updates": [],
    "actions": [],
    "open_questions": [],
    "other": [],
}

for line in lines:
    label, content = split_label(line)
    label_lower = label.lower()

    if label_lower == "proj":
        structured["project"].append(content)
    elif label_lower == "attendees":
        structured["attendees"].append(content)
    elif label_lower == "action":
        structured["actions"].append(content)
    elif label_lower == "open question":
        structured["open_questions"].append(content)
    elif label_lower in ("discussed", "budget", "timeline"):
        structured["updates"].append(f"{label}: {content}")
    else:
        structured["other"].append(f"{label}: {content}")

for key, value in structured.items():
    print(f"{key}: {value}")


project: ['website redesign']
attendees: ['alex, priya, marcus, farah']
updates: ['discussed: homepage layout still not final, priya to send 2 options by fri', 'budget: slightly over, need approval from finance for extra 3k for stock photos', 'timeline: launch pushed from aug 15 to aug 29, dev team needs more QA time']
actions: ['marcus to update timeline doc', "alex to review priya's 2 layout options by mon"]
open_questions: ['do we need a separate mobile design or is responsive enough?']
other: ['action - farah to email finance re: budget approval', 'next meeting: next fri 2pm']


We just turned a flat list of shorthand lines into a **structured dictionary**, grouped by what kind of information each line actually is. This is the single biggest lever in this whole exercise: structured data is dramatically easier for both code *and* AI tools to work with than a wall of loose text.

## Step 4: Build a clean, structured AI prompt from it

In [5]:
def build_prompt(structured):
    prompt_parts = []

    prompt_parts.append("You are helping summarise a work meeting. Here is the structured information:\n")

    if structured["project"]:
        prompt_parts.append(f"Project: {structured['project'][0]}")

    if structured["attendees"]:
        prompt_parts.append(f"Attendees: {structured['attendees'][0]}")

    if structured["updates"]:
        prompt_parts.append("\nKey updates discussed:")
        for update in structured["updates"]:
            prompt_parts.append(f"- {update}")

    if structured["actions"]:
        prompt_parts.append("\nAction items:")
        for action in structured["actions"]:
            prompt_parts.append(f"- {action}")

    if structured["open_questions"]:
        prompt_parts.append("\nOpen questions:")
        for question in structured["open_questions"]:
            prompt_parts.append(f"- {question}")

    prompt_parts.append("\nPlease write this up as: (1) a short paragraph summary, "
                         "(2) a bullet list of action items with clear owners, and "
                         "(3) a one-line flag for anything that still needs a decision.")

    return "\n".join(prompt_parts)

final_prompt = build_prompt(structured)
print(final_prompt)


You are helping summarise a work meeting. Here is the structured information:

Project: website redesign
Attendees: alex, priya, marcus, farah

Key updates discussed:
- discussed: homepage layout still not final, priya to send 2 options by fri
- budget: slightly over, need approval from finance for extra 3k for stock photos
- timeline: launch pushed from aug 15 to aug 29, dev team needs more QA time

Action items:
- marcus to update timeline doc
- alex to review priya's 2 layout options by mon

Open questions:
- do we need a separate mobile design or is responsive enough?

Please write this up as: (1) a short paragraph summary, (2) a bullet list of action items with clear owners, and (3) a one-line flag for anything that still needs a decision.


**Notice what this function is doing:** it's checking `if structured["actions"]:` before adding an "Action items" section. An empty list (`[]`) is treated as `False` in a condition, so this line quietly skips the whole section if there are no action items that day. Small, tidy, no wasted output.

In [6]:
# Save it so you can paste it straight into your AI assistant of choice
with open("generated_prompt.txt", "w") as f:
    f.write(final_prompt)

print("Saved to generated_prompt.txt - copy this into your AI tool of choice.")


Saved to generated_prompt.txt - copy this into your AI tool of choice.


## 🎉 You did it

You just built a bridge between "messy human shorthand" and "reliable AI output" — using nothing but strings, loops, dictionaries, and conditionals. This is the exact skill that separates people who get mediocre, inconsistent results from AI tools from people who get consistently excellent ones: **structuring your input before you hand it over.**

---

## 🧪 Try it yourself

1. Add a new label type to the notes (e.g. `risk -`) and update the code to group it into a new `"risks"` category.
2. Modify `build_prompt()` to also ask the AI to draft a one-paragraph follow-up email to attendees.
3. Take real (anonymised) notes from your own next meeting, in the same loose `label - content` style, and run them through this exact notebook.

## 💡 Where this goes next
This "parse messy input → structure it → hand to an AI with clear instructions" pipeline is exactly how serious AI-powered tools are built behind the scenes. The AI is rarely doing everything end-to-end — good code around it is what makes the output reliable.